# RegimeLab — EURUSD Regime Router
HMM Regime state classification and per-regime strategy router.


In [ ]:
# Papermill Parameter Contract
import os
from pathlib import Path
WORKSPACE_ID = ''
EXPERIMENT_ID = ''
ARTIFACT_DIR = os.environ.get('ARTIFACT_DIR', './artifacts')
RUN_ID = os.environ.get('RUN_ID', 'manual')
STATES = 3
CONFIDENCE_THRESHOLD = 0.65
Path(ARTIFACT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from backend.engine import demo_prices, features, causal_probabilities
from regimelab_sdk import run

df = demo_prices(700)
feat_df = features(df)
probs, model = causal_probabilities(feat_df, states=STATES)
dominant = probs.argmax(axis=1)

state_counts = pd.Series(dominant).value_counts().to_dict()
print('Regime distribution:', state_counts)

run.log_metric('regime_states', STATES)
run.log_metric('dominant_regime_ratio', round(float(max(state_counts.values()) / len(dominant)), 3))
regimes_csv = Path(ARTIFACT_DIR) / 'regime_assignments.csv'
pd.DataFrame({'dominant_regime': dominant}).to_csv(regimes_csv, index=False)
run.log_artifact(regimes_csv)
run.log_message('Regime Router model generated successfully.')
